# 02 — Run CellPhoneDB v5 (35 runs)

Executes the CellPhoneDB v5.0.1 statistical analysis for every input directory built by
`01_build_cpdb_inputs.ipynb`.

**Run order (35 runs):** 24 per-patient -> pooled seed-0 Good -> pooled seed-0 Poor ->
all-samples -> pooled seeds 1-4 Good/Poor (8 runs).

## Deliberate change from every prior run: `result_precision=6`

Prior runs used `result_precision=3`. On the 0-100 interaction-score scale, 3 decimals
quantise at 1e-3 and manufacture ties that degrade the downstream rank tests. This notebook
uses `result_precision=6` (`config.yaml: cpdb_result_precision`).

**Documented CellPhoneDB 5.0.1 limitation — verified by reading the installed package.**
`result_precision` is applied to `means`, `significant_means`, `deconvoluted` and
`deconvoluted_percents` only
(`cellphonedb/src/core/methods/cpdb_statistical_analysis_complex_method.py`, `.round(result_precision)`).
It is **not** applied to `interaction_scores`: `cellphonedb/utils/scoring_utils._get_lr_scores`
contains a hardcoded `lr_outer = np.around(lr_outer, 3)`, so interaction scores are always
returned at 3 decimals on a 0-100 scale regardless of `result_precision`. Gate 2 asserts this
explicitly rather than silently assuming the requested precision was honoured, and records it
in `logs/02_precision_note.json` so downstream tie handling is an informed choice, not an
accident.

## Resumability contract

A run counts as complete **only** if all required outputs exist, are non-empty, parse,
carry the expected metadata and cell-pair columns, hash-match, **and** a completion marker
exists whose config hash, database hash, CPDB parameters and input hashes all match the
current configuration. Existence of a single file never causes a skip. Runs are written to a
temporary directory and atomically published on success.

## Do not hand-edit outputs

Nothing here writes into `cpdb_results_*`, `t1/t2/t3_*` or `cellphoneDB_copy_backup/`.

In [ ]:
import os
# Project data root. Override with the ACC_DATA_ROOT environment variable.
ACC_DATA_ROOT = os.environ.get("ACC_DATA_ROOT", os.path.expanduser("~/acc_data"))


## 1. Configuration (single config cell — no hidden state below this point)

In [ ]:
# ============================================================
# CONFIG CELL — the only place values are declared.
# ============================================================
import inspect
import json
import os
import shutil
import subprocess
import sys
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore")

CODE_ROOT = Path(
    f"{ACC_DATA_ROOT}/scripts/named_scripts/cellphoneDB_scripts/clinical_outcome_v2"
)
CONFIG_PATH = CODE_ROOT / "config.yaml"
assert CODE_ROOT.is_dir(), CODE_ROOT
assert CONFIG_PATH.is_file(), CONFIG_PATH

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

import utils  # noqa: E402  (shared helpers written by 00_/utils.py — never reimplemented here)
from utils import (  # noqa: E402
    atomic_publish,
    config_hash,
    load_config,
    log_jsonl,
    parse_cellpair_columns,
    run_is_complete,
    sha256_file,
    write_success_json,
)

CFG = load_config(CONFIG_PATH)
CFG_HASH = config_hash(CFG)

OUTPUT_ROOT = Path(CFG["output_root"])
INPUT_ROOT = OUTPUT_ROOT / "inputs"
RUN_ROOT = OUTPUT_ROOT / "cpdb_runs"
LOG_DIR = OUTPUT_ROOT / "logs"
CPDB_DB = Path(CFG["cpdb_database"])

RUN_LOG = LOG_DIR / "02_run_log.jsonl"
SUCCESS_RECORD = LOG_DIR / "02_success_record.json"
PRECISION_NOTE = LOG_DIR / "02_precision_note.json"

# Locked CellPhoneDB call parameters. Any change invalidates every completion marker.
CPDB_PARAMS = {
    "counts_data": "hgnc_symbol",
    "score_interactions": True,
    "iterations": int(CFG["cpdb_iterations"]),
    "threshold": float(CFG["cpdb_expression_fraction"]),
    "threads": int(CFG["threads"]),
    "debug_seed": int(CFG["cpdb_debug_seed"]),
    "result_precision": int(CFG["cpdb_result_precision"]),
    "pvalue": float(CFG["cpdb_nominal_pvalue"]),
    "output_suffix": None,
}

# Files 01_ writes into every input directory.
INPUT_FILES = ("counts.h5ad", "meta.tsv", "retained_celltypes.json")

# CellPhoneDB 5.0.1 returns these keys (verified against the installed package):
#   deconvoluted, deconvoluted_percents, means, pvalues, significant_means,
#   CellSign_active_interactions, CellSign_active_interactions_deconvoluted
#   + interaction_scores (added when score_interactions=True).
# The two CellSign frames are empty unless an active-TF file is supplied, and empty frames
# are not written. Only the four below are contractually required.
REQUIRED_OUTPUTS = ("interaction_scores.csv", "means.csv", "pvalues.csv", "significant_means.csv")
OPTIONAL_OUTPUTS = ("deconvoluted.csv", "deconvoluted_percents.csv")

# Metadata (non cell-pair) columns every result table must carry.
REQUIRED_META_COLUMNS = (
    "id_cp_interaction", "interacting_pair", "partner_a", "partner_b", "gene_a", "gene_b",
    "secreted", "receptor_a", "receptor_b", "annotation_strategy", "is_integrin",
    "directionality", "classification",
)

# Tables laid out as (metadata columns + one column per directional cell pair).
PAIRED_OUTPUTS = ("interaction_scores.csv", "means.csv", "pvalues.csv", "significant_means.csv")

CELLPAIR_SEPARATOR = "|"

# Cost estimate constants, measured from the existing prect_3 / annot_6 result folders.
EST_SECONDS_PER_PATIENT_RUN = 15.0
EST_SECONDS_PER_POOLED_RUN = 5.0 * 60.0
EST_SECONDS_ALL_SAMPLES_RUN = 8.0 * 60.0
EST_MB_PER_RUN_DIR = 800.0

# Gate 2 expectation from the locked plan: the pooled Good/Poor comparison retains 15 types.
EXPECTED_POOLED_N_CELLTYPES = 15

for d in (RUN_ROOT, LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)

assert INPUT_ROOT.is_dir(), f"01_ has not produced inputs at {INPUT_ROOT}"
assert CPDB_DB.is_file(), CPDB_DB

## 2. Shared-API contract check

The exact positional signature of every shared helper is pinned. If `utils.py` changes shape,
this notebook stops here rather than silently misbinding an argument during a 45-minute run.

In [ ]:
# ============================================================
# API contract — pin the shared helpers' signatures before doing expensive work.
# ============================================================
EXPECTED_SIGNATURES = {
    "load_config": ["path"],
    "config_hash": ["cfg"],
    "sha256_file": ["path", "chunk_size"],
    "write_success_json": ["run_dir", "fields"],          # fields is **kwargs
    "run_is_complete": ["run_dir", "required_files", "cfg_hash"],
    "atomic_publish": ["tmp_dir", "final_dir"],
    "log_jsonl": ["path", "record"],
    "parse_cellpair_columns": ["cols", "known_celltypes"],
}
UTILS_SIGNATURES = {}
for name, expected in EXPECTED_SIGNATURES.items():
    fn = getattr(utils, name, None)
    assert callable(fn), f"utils.py does not expose {name}()"
    actual = list(inspect.signature(fn).parameters)
    assert actual == expected, (
        f"utils.{name}{inspect.signature(fn)} no longer matches the pinned contract "
        f"{expected}. Update this notebook deliberately; do not adapt silently."
    )
    UTILS_SIGNATURES[name] = str(inspect.signature(fn))

MARKER_NAME = utils.SUCCESS_FILENAME
assert MARKER_NAME.endswith(".json"), MARKER_NAME
assert utils.PAIR_SEP == CELLPAIR_SEPARATOR, (
    f"utils.PAIR_SEP is {utils.PAIR_SEP!r} but this notebook assumes {CELLPAIR_SEPARATOR!r}"
)


def parse_pairs(columns, known_celltypes):
    '''utils.parse_cellpair_columns, with the arity invariant made explicit.

    It skips metadata columns and raises on any '|' column whose halves are not known cell
    types, so passing only the pair columns must return exactly one pair per column.
    '''
    pairs = parse_cellpair_columns(list(columns), list(known_celltypes))
    assert len(pairs) == len(columns), (
        f"parse_cellpair_columns returned {len(pairs)} pairs for {len(columns)} cell-pair columns"
    )
    return pairs


CPDB_DB_HASH = sha256_file(CPDB_DB)
UTILS_SOURCE_HASH = sha256_file(CODE_ROOT / "utils.py")
CONFIG_FILE_HASH = sha256_file(CONFIG_PATH)
UTILS_SIGNATURES

## 3. Environment and CellPhoneDB import

In [ ]:
import cellphonedb  # noqa: E402
from cellphonedb.src.core.methods import cpdb_statistical_analysis_method  # noqa: E402

CPDB_VERSION = getattr(cellphonedb, "__version__", None)
if CPDB_VERSION is None:
    CPDB_VERSION = subprocess.run(
        [sys.executable, "-m", "pip", "show", "cellphonedb"],
        capture_output=True, text=True, check=True,
    ).stdout.split("Version:")[1].split("\n")[0].strip()

assert CPDB_VERSION == "5.0.1", f"Expected CellPhoneDB 5.0.1, found {CPDB_VERSION}"

# `call` must still accept every locked parameter name; a version drift breaks the contract.
CALL_SIG = inspect.signature(cpdb_statistical_analysis_method.call)
for param in ("cpdb_file_path", "meta_file_path", "counts_file_path", "output_path", *CPDB_PARAMS):
    assert param in CALL_SIG.parameters, f"cpdb_statistical_analysis_method.call lacks '{param}'"

ENVIRONMENT = {
    "python": sys.version.split()[0],
    "executable": sys.executable,
    "cellphonedb": CPDB_VERSION,
    "pandas": pd.__version__,
    "cpu_count": os.cpu_count(),
}
assert ENVIRONMENT["cpu_count"] >= CPDB_PARAMS["threads"], (
    f"threads={CPDB_PARAMS['threads']} exceeds {ENVIRONMENT['cpu_count']} available cores"
)
ENVIRONMENT

## 4. Run inventory

Derived entirely from the input directories written by `01_`. No patient list is hardcoded
anywhere in this notebook.

In [ ]:
# ============================================================
# Build the 35-run inventory from the input tree, in the locked execution order.
# ============================================================
def _load_retained(input_dir: Path):
    payload = json.loads((input_dir / "retained_celltypes.json").read_text())
    if isinstance(payload, dict):
        for key in ("retained_celltypes", "retained", "cell_types", "celltypes"):
            if key in payload:
                payload = payload[key]
                break
    assert isinstance(payload, list) and payload, f"Unreadable retained_celltypes.json in {input_dir}"
    labels = sorted(str(x) for x in payload)
    assert len(set(labels)) == len(labels), f"Duplicate cell types in {input_dir}"
    for label in labels:
        assert CELLPAIR_SEPARATOR not in label, (
            f"Cell-type label {label!r} contains {CELLPAIR_SEPARATOR!r}; cell-pair column "
            "parsing would be ambiguous"
        )
    return labels


def make_spec(name, family, input_dir: Path, run_dir: Path):
    input_dir = Path(input_dir)
    for fname in INPUT_FILES:
        f = input_dir / fname
        assert f.is_file() and f.stat().st_size > 0, f"Missing/empty input {f}"
    retained = _load_retained(input_dir)
    return {
        "name": name,
        "family": family,
        "input_dir": input_dir,
        "run_dir": Path(run_dir),
        "retained_celltypes": retained,
        "expected_pairs": tuple(
            f"{a}{CELLPAIR_SEPARATOR}{b}" for a in retained for b in retained
        ),
        "input_hashes": {fname: sha256_file(input_dir / fname) for fname in INPUT_FILES},
    }


SEEDS = [int(s) for s in CFG["pooled_downsample_seeds"]]
assert SEEDS[0] == 0, f"seed 0 must be the main pooled run; got {SEEDS}"
GROUPS = ("good", "poor")

patient_dirs = sorted(p for p in (INPUT_ROOT / "per_patient").iterdir() if p.is_dir())
assert patient_dirs, "No per-patient input directories found"

SPECS = []
# 1. per-patient
for pdir in patient_dirs:
    SPECS.append(make_spec(pdir.name, "per_patient", pdir, RUN_ROOT / "per_patient" / pdir.name))
# 2. pooled seed 0, good then poor
for grp in GROUPS:
    SPECS.append(make_spec(f"pooled_seed0_{grp}", "pooled_seed0",
                           INPUT_ROOT / "pooled_seed0" / grp, RUN_ROOT / "pooled" / grp))
# 3. all samples
SPECS.append(make_spec("all", "all", INPUT_ROOT / "all", RUN_ROOT / "all"))
# 4. pooled sensitivity seeds
for seed in SEEDS[1:]:
    for grp in GROUPS:
        SPECS.append(make_spec(f"pooled_seed{seed}_{grp}", f"pooled_seed{seed}",
                               INPUT_ROOT / f"pooled_seed{seed}" / grp,
                               RUN_ROOT / f"pooled_seed{seed}" / grp))

N_PATIENTS = len(patient_dirs)
N_EXPECTED_RUNS = N_PATIENTS + 2 * len(SEEDS) + 1
assert len(SPECS) == N_EXPECTED_RUNS == 35, (
    f"Expected 35 runs ({N_PATIENTS} patients + {2 * len(SEEDS)} pooled + 1 all-samples), "
    f"built {len(SPECS)}"
)
assert len({s["name"] for s in SPECS}) == len(SPECS), "Duplicate run names"
assert len({s["run_dir"] for s in SPECS}) == len(SPECS), "Duplicate run directories"
SPEC_BY_NAME = {s["name"]: s for s in SPECS}

# No run may write anywhere except under cpdb_runs/.
for s in SPECS:
    assert RUN_ROOT in s["run_dir"].parents, s["run_dir"]
    for forbidden in ("cpdb_results_", "t1_", "t2_", "t3_", "cellphoneDB_copy_backup"):
        assert forbidden not in str(s["run_dir"]), s["run_dir"]

pd.DataFrame(
    [{"name": s["name"], "family": s["family"], "n_celltypes": len(s["retained_celltypes"]),
      "n_pairs": len(s["expected_pairs"]), "run_dir": str(s["run_dir"])} for s in SPECS]
)

## 5. Cost estimate

In [ ]:
# ============================================================
# Cost estimate (printed for human sign-off before the expensive loop).
# ============================================================
def _estimate_seconds(spec):
    if spec["family"] == "per_patient":
        return EST_SECONDS_PER_PATIENT_RUN
    if spec["family"] == "all":
        return EST_SECONDS_ALL_SAMPLES_RUN
    return EST_SECONDS_PER_POOLED_RUN


est = pd.DataFrame([{"family": s["family"], "seconds": _estimate_seconds(s)} for s in SPECS])
est_total_min = est["seconds"].sum() / 60.0
est_total_gb = len(SPECS) * EST_MB_PER_RUN_DIR / 1024.0
free_gb = shutil.disk_usage(OUTPUT_ROOT).free / 1024 ** 3

print("CellPhoneDB cost estimate")
print("-" * 64)
print(est.groupby("family")["seconds"].agg(["count", "sum"]).assign(
    minutes=lambda d: d["sum"] / 60.0).to_string())
print("-" * 64)
print(f"runs                 : {len(SPECS)}")
print(f"estimated wall time  : {est_total_min:.0f} min (measured basis: 24 per-patient runs "
      f"~6.5 min total; a pooled run ~5 min)")
print(f"disk budget (upper)  : {est_total_gb:.1f} GB (~{EST_MB_PER_RUN_DIR:.0f} MB/run dir)")
print(f"free disk at target  : {free_gb:.1f} GB")
print(f"threads per run      : {CPDB_PARAMS['threads']} of {ENVIRONMENT['cpu_count']} cores")
print(f"result_precision     : {CPDB_PARAMS['result_precision']} "
      f"(interaction_scores are fixed at 3 dp by CellPhoneDB — see header)")
print()
print("Note on the disk budget: the ~800 MB/dir figure comes from the legacy result folders,")
print("which stored counts.h5ad alongside the CSVs. Here counts.h5ad lives once under")
print(f"{INPUT_ROOT} and is NOT copied into cpdb_runs/, so real run directories hold only the")
print("result CSVs plus CellPhoneDB's own duplicate .txt exports (order 10-30 MB each). The")
print("800 MB figure is retained as a deliberately conservative upper bound.")

assert free_gb > 2.0 * est_total_gb, (
    f"Only {free_gb:.1f} GB free for a conservative {est_total_gb:.1f} GB budget"
)

## 6. Completion checking

`utils.run_is_complete` is the shared gate. This notebook adds the structural checks the
resumability contract requires — required outputs present, non-empty, parseable, correct
metadata and cell-pair columns, and hash-identical to what the marker recorded — and both
must agree before a run is skipped.

In [ ]:
# ============================================================
# Marker payload + completeness verification.
# ============================================================
def marker_path(run_dir: Path):
    return Path(run_dir) / MARKER_NAME


def expected_marker_fields(spec):
    '''The identity of a run. Any difference here forces a rerun.

    'config_hash' is the key utils.run_is_complete inspects, so it is spelled exactly that.
    '''
    return {
        "run_name": spec["name"],
        "config_hash": CFG_HASH,
        "config_file_sha256": CONFIG_FILE_HASH,
        "database_sha256": CPDB_DB_HASH,
        "cpdb_version": CPDB_VERSION,
        "cpdb_params": CPDB_PARAMS,
        "input_hashes": spec["input_hashes"],
    }


def _verify_table(path: Path, spec, marker):
    '''Parse one published table and check its shape/columns against the marker.'''
    if not path.is_file():
        return f"missing output {path.name}"
    if path.stat().st_size == 0:
        return f"empty output {path.name}"
    recorded = (marker.get("output_hashes") or {}).get(path.name)
    if recorded is None:
        return f"marker has no hash for {path.name}"
    if sha256_file(path) != recorded:
        return f"hash mismatch for {path.name}"
    try:
        df = pd.read_csv(path)
    except Exception as exc:  # noqa: BLE001 - unparseable output must not count as complete
        return f"unparseable {path.name}: {exc}"
    if df.empty:
        return f"zero rows in {path.name}"
    shape = (marker.get("shapes") or {}).get(path.name)
    if shape is not None and list(df.shape) != list(shape):
        return f"shape {df.shape} != recorded {tuple(shape)} for {path.name}"
    if path.name in PAIRED_OUTPUTS:
        missing_meta = [c for c in REQUIRED_META_COLUMNS if c not in df.columns]
        if missing_meta:
            return f"{path.name} missing metadata columns {missing_meta}"
        pair_cols = [c for c in df.columns if CELLPAIR_SEPARATOR in c]
        if set(pair_cols) != set(spec["expected_pairs"]):
            return (f"{path.name} cell-pair columns do not match "
                    f"{len(spec['retained_celltypes'])}^2 retained types "
                    f"({len(pair_cols)} found, {len(spec['expected_pairs'])} expected)")
        try:
            parsed = parse_pairs(pair_cols, spec["retained_celltypes"])
        except AssertionError as exc:
            return f"{path.name} cell-pair parsing failed: {exc}"
        allowed = set(spec["retained_celltypes"])
        bad = {t for pair in parsed for t in pair if t not in allowed}
        if bad:
            return f"{path.name} cell-pair columns contain unknown cell types {sorted(bad)}"
    return None


def check_complete(spec):
    '''Return (is_complete, reason). Never skips on the existence of a single file.'''
    run_dir = spec["run_dir"]
    if not run_dir.is_dir():
        return False, "no run directory"
    mpath = marker_path(run_dir)
    if not mpath.is_file():
        return False, "no completion marker"
    try:
        marker = json.loads(mpath.read_text())
    except Exception as exc:  # noqa: BLE001
        return False, f"unreadable marker: {exc}"

    for key, want in expected_marker_fields(spec).items():
        if marker.get(key) != want:
            return False, f"marker field {key!r} differs from current configuration"
    if marker.get("status") != "success":
        return False, f"marker status {marker.get('status')!r}"

    for fname in REQUIRED_OUTPUTS:
        problem = _verify_table(run_dir / fname, spec, marker)
        if problem:
            return False, problem

    if not run_is_complete(run_dir, list(REQUIRED_OUTPUTS), CFG_HASH):
        return False, "utils.run_is_complete rejected the run"
    return True, "complete"

## 7. Run one CellPhoneDB analysis (temp dir → atomic publish → marker last)

In [ ]:
# ============================================================
# Single-run execution.
# ============================================================
def run_one(spec):
    run_dir = spec["run_dir"]
    tmp_dir = run_dir.parent / f".{run_dir.name}.inprogress"
    run_dir.parent.mkdir(parents=True, exist_ok=True)
    if tmp_dir.exists():
        shutil.rmtree(tmp_dir)
    tmp_dir.mkdir(parents=True)

    t0 = time.time()
    started = datetime.now(timezone.utc).isoformat()

    results = cpdb_statistical_analysis_method.call(
        cpdb_file_path=str(CPDB_DB),
        meta_file_path=str(spec["input_dir"] / "meta.tsv"),
        counts_file_path=str(spec["input_dir"] / "counts.h5ad"),
        output_path=str(tmp_dir),
        **CPDB_PARAMS,
    )

    assert isinstance(results, dict), f"CellPhoneDB returned {type(results)}"
    returned_keys = sorted(results)
    written, shapes, empty_keys = {}, {}, []
    for key, df in results.items():
        if not isinstance(df, pd.DataFrame):
            continue
        if df.empty:
            empty_keys.append(key)
            continue
        out = tmp_dir / f"{key}.csv"
        df.to_csv(out, index=False)
        written[out.name] = out
        shapes[out.name] = list(df.shape)

    missing_required = [f for f in REQUIRED_OUTPUTS if f not in written]
    assert not missing_required, (
        f"CellPhoneDB did not return {missing_required}; returned keys={returned_keys}, "
        f"empty frames={empty_keys}"
    )

    # Structural validation before publishing anything.
    for fname in REQUIRED_OUTPUTS:
        df = pd.read_csv(written[fname])
        assert not df.empty, f"{fname} has zero rows"
        if fname in PAIRED_OUTPUTS:
            missing_meta = [c for c in REQUIRED_META_COLUMNS if c not in df.columns]
            assert not missing_meta, f"{fname} missing metadata columns {missing_meta}"
            pair_cols = [c for c in df.columns if CELLPAIR_SEPARATOR in c]
            assert set(pair_cols) == set(spec["expected_pairs"]), (
                f"{fname}: {len(pair_cols)} cell-pair columns, expected "
                f"{len(spec['expected_pairs'])} from {len(spec['retained_celltypes'])} retained types"
            )
        assert df["id_cp_interaction"].is_unique, f"{fname} has duplicate id_cp_interaction"

    output_hashes = {name: sha256_file(path) for name, path in written.items()}
    elapsed = time.time() - t0
    finished = datetime.now(timezone.utc).isoformat()

    marker = dict(expected_marker_fields(spec))
    marker.update({
        "status": "success",
        "started_utc": started,
        "finished_utc": finished,
        "elapsed_seconds": round(elapsed, 3),
        "environment": ENVIRONMENT,
        "utils_sha256": UTILS_SOURCE_HASH,
        "retained_celltypes": spec["retained_celltypes"],
        "n_celltypes": len(spec["retained_celltypes"]),
        "n_cell_pairs": len(spec["expected_pairs"]),
        "returned_keys": returned_keys,
        "empty_returned_keys": sorted(empty_keys),
        "written_outputs": sorted(written),
        "shapes": shapes,
        "output_hashes": output_hashes,
        "marker_written_last": True,
    })

    # Marker is the last thing written into the temp directory, then the whole directory is
    # published atomically, so a marker can never become visible ahead of its outputs.
    # utils.write_success_json takes the fields as **kwargs, not as one payload object.
    write_success_json(tmp_dir, **marker)
    assert marker_path(tmp_dir).is_file(), "write_success_json did not create the marker"
    written_marker = json.loads(marker_path(tmp_dir).read_text())
    assert written_marker["config_hash"] == CFG_HASH, (
        "The marker does not carry config_hash at the top level; utils.run_is_complete "
        "would reject this run."
    )

    # utils.atomic_publish displaces any pre-existing final directory itself and only deletes
    # it once the new one is in place, so a failed run directory is superseded, never merged.
    atomic_publish(tmp_dir, run_dir)
    assert run_dir.is_dir() and not tmp_dir.exists(), f"atomic_publish left {tmp_dir}"
    return marker

## 8. Execution loop

Independent runs continue past failures; the notebook fails at Gate 2 if anything failed.

In [ ]:
# ============================================================
# 35-run loop. run_log.jsonl is flushed after every single run.
# ============================================================
RESULTS, FAILURES = [], []
loop_started = datetime.now(timezone.utc).isoformat()

for i, spec in enumerate(SPECS, start=1):
    complete, reason = check_complete(spec)
    if complete:
        record = {"event": "skip", "run": spec["name"], "family": spec["family"],
                  "reason": reason, "index": i, "n_runs": len(SPECS),
                  "timestamp_utc": datetime.now(timezone.utc).isoformat()}
        RESULTS.append({"run": spec["name"], "status": "skipped", "elapsed_seconds": 0.0})
        log_jsonl(RUN_LOG, record)
        print(f"[{i:2d}/{len(SPECS)}] {spec['name']:<24s} SKIP (already complete)")
        continue

    print(f"[{i:2d}/{len(SPECS)}] {spec['name']:<24s} RUN  ({reason})", flush=True)
    t0 = time.time()
    try:
        marker = run_one(spec)
        record = {"event": "run", "status": "success", "run": spec["name"],
                  "family": spec["family"], "trigger": reason,
                  "elapsed_seconds": marker["elapsed_seconds"],
                  "n_celltypes": marker["n_celltypes"], "shapes": marker["shapes"],
                  "returned_keys": marker["returned_keys"],
                  "empty_returned_keys": marker["empty_returned_keys"],
                  "index": i, "timestamp_utc": datetime.now(timezone.utc).isoformat()}
        RESULTS.append({"run": spec["name"], "status": "success",
                        "elapsed_seconds": marker["elapsed_seconds"]})
    except Exception as exc:  # noqa: BLE001 - one failure must not abort the other 34 runs
        import traceback
        elapsed = time.time() - t0
        # Leave no half-written temp directory behind for the next attempt to trip over.
        shutil.rmtree(spec["run_dir"].parent / f".{spec['run_dir'].name}.inprogress",
                      ignore_errors=True)
        FAILURES.append({"run": spec["name"], "error": f"{type(exc).__name__}: {exc}"})
        record = {"event": "run", "status": "failed", "run": spec["name"],
                  "family": spec["family"], "trigger": reason,
                  "elapsed_seconds": round(elapsed, 3),
                  "error_type": type(exc).__name__, "error": str(exc),
                  "traceback": traceback.format_exc(),
                  "index": i, "timestamp_utc": datetime.now(timezone.utc).isoformat()}
        RESULTS.append({"run": spec["name"], "status": "failed",
                        "elapsed_seconds": round(elapsed, 3)})
        print(f"    FAILED: {type(exc).__name__}: {exc}", flush=True)
    log_jsonl(RUN_LOG, record)

loop_finished = datetime.now(timezone.utc).isoformat()
RUN_SUMMARY = pd.DataFrame(RESULTS)

assert len(RESULTS) == len(SPECS), f"{len(RESULTS)} outcomes recorded for {len(SPECS)} runs"
assert RUN_LOG.is_file(), f"{RUN_LOG} was never written"
n_log_lines = sum(1 for line in RUN_LOG.read_text().splitlines() if line.strip())
assert n_log_lines >= len(SPECS), (
    f"{RUN_LOG} holds {n_log_lines} records for {len(SPECS)} runs — it was not flushed per run"
)
# No temp directories may survive the loop.
leftovers = list(RUN_ROOT.rglob(".*.inprogress")) + list(RUN_ROOT.rglob("*.superseded.*"))
assert not leftovers, f"Temporary run directories left behind: {leftovers}"

RUN_SUMMARY["status"].value_counts()

## 9. Gate 2

1. 35 completion markers exist and every recorded hash matches the current configuration.
2. Re-running the notebook is a no-op: the completeness decision now returns *skip* for all 35.
3. The pooled Good and Poor `pvalues.csv` headers cover the same 15 cell types (225 pairs).
4. Precision: `means`/`significant_means` carry more than 3 decimals, and `interaction_scores`
   is confirmed to be pinned at 3 decimals by CellPhoneDB itself.

In [ ]:
# ============================================================
# GATE 2.1 — every run failed-safe and 35 verified markers exist.
# ============================================================
assert not FAILURES, f"{len(FAILURES)} CellPhoneDB run(s) failed: {FAILURES}"

recheck = {spec["name"]: check_complete(spec) for spec in SPECS}
incomplete = {k: v[1] for k, v in recheck.items() if not v[0]}
assert not incomplete, f"Runs not verifiably complete after execution: {incomplete}"
assert len(recheck) == 35 and all(v[0] for v in recheck.values())

markers = {}
for spec in SPECS:
    mpath = marker_path(spec["run_dir"])
    assert mpath.is_file(), f"No completion marker for {spec['name']}"
    markers[spec["name"]] = json.loads(mpath.read_text())

assert len(markers) == 35, f"{len(markers)} markers, expected 35"
for name, m in markers.items():
    assert m["config_hash"] == CFG_HASH, name
    assert m["config_file_sha256"] == CONFIG_FILE_HASH, name
    assert m["database_sha256"] == CPDB_DB_HASH, name
    assert m["cpdb_version"] == CPDB_VERSION, name
    assert m["cpdb_params"] == CPDB_PARAMS, name
    assert m["cpdb_params"]["result_precision"] == int(CFG["cpdb_result_precision"]) == 6, name
    assert m["input_hashes"] == SPEC_BY_NAME[name]["input_hashes"], name
    assert m["status"] == "success", name
    assert m["marker_written_last"] is True, name

In [ ]:
# Re-verify every published output hash against its marker (no shortcuts).
for name, m in markers.items():
    run_dir = SPEC_BY_NAME[name]["run_dir"]
    for fname, digest in m["output_hashes"].items():
        path = run_dir / fname
        assert path.is_file(), f"{name}: {fname} recorded in marker but absent"
        assert sha256_file(path) == digest, f"{name}: {fname} changed since the marker was written"
    for fname in REQUIRED_OUTPUTS:
        assert fname in m["output_hashes"], f"{name}: required output {fname} not in marker"

In [ ]:
# ============================================================
# GATE 2.2 — re-running is a no-op.
# ============================================================
second_pass = [(s["name"], *check_complete(s)) for s in SPECS]
would_rerun = [(n, r) for n, ok, r in second_pass if not ok]
assert not would_rerun, f"Second pass would re-execute: {would_rerun}"
assert len(second_pass) == 35

# And the published outputs are byte-identical to the first pass (nothing was rewritten).
post_hashes = {
    n: {f: sha256_file(SPEC_BY_NAME[n]["run_dir"] / f) for f in m["output_hashes"]}
    for n, m in markers.items()
}
assert post_hashes == {n: m["output_hashes"] for n, m in markers.items()}, (
    "Outputs changed during the idempotency check"
)

In [ ]:
# ============================================================
# GATE 2.3 — pooled Good/Poor share one 15-cell-type, 225-pair grid.
# ============================================================
pooled_good = RUN_ROOT / "pooled" / "good" / "pvalues.csv"
pooled_poor = RUN_ROOT / "pooled" / "poor" / "pvalues.csv"

pooled_retained = {
    grp: SPEC_BY_NAME[f"pooled_seed0_{grp}"]["retained_celltypes"] for grp in GROUPS
}
pair_sets, type_sets = {}, {}
for label, path in (("good", pooled_good), ("poor", pooled_poor)):
    header = pd.read_csv(path, nrows=0)
    pairs = [c for c in header.columns if CELLPAIR_SEPARATOR in c]
    parsed = parse_pairs(pairs, pooled_retained[label])
    pair_sets[label] = set(pairs)
    type_sets[label] = {t for pair in parsed for t in pair}
    # Header-derived types must equal what 01_ recorded as retained for that group.
    assert type_sets[label] == set(pooled_retained[label]), (
        f"pooled {label}: pvalues.csv header types differ from retained_celltypes.json"
    )

assert type_sets["good"] == type_sets["poor"], (
    "Pooled Good/Poor cell-type sets differ: "
    f"good-only={sorted(type_sets['good'] - type_sets['poor'])}, "
    f"poor-only={sorted(type_sets['poor'] - type_sets['good'])}"
)
N_POOLED_TYPES = len(type_sets["good"])
assert N_POOLED_TYPES == EXPECTED_POOLED_N_CELLTYPES, (
    f"Pooled runs cover {N_POOLED_TYPES} cell types, plan expects {EXPECTED_POOLED_N_CELLTYPES}"
)
assert pair_sets["good"] == pair_sets["poor"], "Pooled Good/Poor cell-pair columns differ"
assert len(pair_sets["good"]) == N_POOLED_TYPES ** 2 == 225, (
    f"{len(pair_sets['good'])} pooled cell pairs, expected {N_POOLED_TYPES ** 2}"
)

# Sensitivity seeds must sit on the same grid, otherwise they are not comparable to seed 0.
for seed in SEEDS[1:]:
    for grp in GROUPS:
        header = pd.read_csv(RUN_ROOT / f"pooled_seed{seed}" / grp / "pvalues.csv", nrows=0)
        pairs = {c for c in header.columns if CELLPAIR_SEPARATOR in c}
        assert pairs == pair_sets["good"], f"pooled_seed{seed}/{grp} uses a different cell-pair grid"

sorted(type_sets["good"])

In [ ]:
# ============================================================
# GATE 2.4 — precision actually delivered, per table.
# ============================================================
def _decimal_evidence(path, sample_rows=4000):
    df = pd.read_csv(path, nrows=sample_rows)
    num = df[[c for c in df.columns if CELLPAIR_SEPARATOR in c]].apply(
        pd.to_numeric, errors="coerce")
    vals = num.to_numpy(dtype="float64").ravel()
    vals = vals[pd.notna(vals)]
    assert vals.size, f"No numeric cell-pair values in {path}"
    beyond_3dp = (vals != vals.round(3)).sum()
    beyond_6dp = (vals != vals.round(6)).sum()
    return {"n_values": int(vals.size), "n_beyond_3dp": int(beyond_3dp),
            "n_beyond_6dp": int(beyond_6dp), "max": float(vals.max())}


PRECISION_EVIDENCE = {}
for label, run_dir in (("pooled_good", RUN_ROOT / "pooled" / "good"),
                       ("all", RUN_ROOT / "all"),
                       (patient_dirs[0].name, RUN_ROOT / "per_patient" / patient_dirs[0].name)):
    for fname in ("means.csv", "significant_means.csv", "interaction_scores.csv"):
        PRECISION_EVIDENCE[f"{label}/{fname}"] = _decimal_evidence(run_dir / fname)

# means / significant_means must show resolution finer than 3 dp, i.e. result_precision=6 took.
for key, ev in PRECISION_EVIDENCE.items():
    if key.endswith("means.csv"):
        assert ev["n_beyond_3dp"] > 0, (
            f"{key} carries no value beyond 3 decimals — result_precision=6 was not applied"
        )
        assert ev["n_beyond_6dp"] == 0, f"{key} carries values beyond 6 decimals"

# interaction_scores: CellPhoneDB 5.0.1 hardcodes np.around(..., 3) in
# utils/scoring_utils._get_lr_scores, so result_precision cannot raise its resolution.
# Assert the observed behaviour rather than assume the requested precision.
for key, ev in PRECISION_EVIDENCE.items():
    if key.endswith("interaction_scores.csv"):
        assert ev["n_beyond_3dp"] == 0, (
            f"{key}: interaction scores now carry >3 decimals — CellPhoneDB behaviour changed, "
            "revisit the tie-handling assumptions in 04_"
        )
        assert ev["max"] <= 100.0 + 1e-9, f"{key}: score above the 0-100 scale"

PRECISION_NOTE.write_text(json.dumps({
    "cellphonedb_version": CPDB_VERSION,
    "requested_result_precision": CPDB_PARAMS["result_precision"],
    "applies_to": ["means", "significant_means", "deconvoluted", "deconvoluted_percents"],
    "does_not_apply_to": ["interaction_scores"],
    "reason": ("cellphonedb/utils/scoring_utils._get_lr_scores hardcodes "
               "lr_outer = np.around(lr_outer, 3); result_precision is never passed into "
               "the interaction-scoring path."),
    "consequence": ("Interaction scores are quantised at 1e-3 on a 0-100 scale (upper_range=10 "
                    "per participant, product in [0,100]). Ties are therefore expected in the "
                    "patient-level rank tests; 04_ must use tie-corrected statistics."),
    "evidence": PRECISION_EVIDENCE,
    "generated_utc": datetime.now(timezone.utc).isoformat(),
}, indent=2))
pd.DataFrame(PRECISION_EVIDENCE).T

## 10. Machine-readable success record

In [ ]:
# ============================================================
# Final success record for 02_.
# ============================================================
elapsed_by_run = {n: m["elapsed_seconds"] for n, m in markers.items()}
record = {
    "notebook": "02_run_cellphonedb.ipynb",
    "status": "success",
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "loop_started_utc": loop_started,
    "loop_finished_utc": loop_finished,
    "config_hash": CFG_HASH,
    "config_file_sha256": CONFIG_FILE_HASH,
    "utils_sha256": UTILS_SOURCE_HASH,
    "database_sha256": CPDB_DB_HASH,
    "environment": ENVIRONMENT,
    "cpdb_params": CPDB_PARAMS,
    "utils_signatures": UTILS_SIGNATURES,
    "n_runs": len(SPECS),
    "n_patients": N_PATIENTS,
    "pooled_seeds": SEEDS,
    "n_pooled_celltypes": int(N_POOLED_TYPES),
    "n_pooled_cell_pairs": int(len(pair_sets["good"])),
    "cpdb_returned_keys": sorted({k for m in markers.values() for k in m["returned_keys"]}),
    "cpdb_empty_returned_keys": sorted({k for m in markers.values()
                                        for k in m["empty_returned_keys"]}),
    "required_outputs": list(REQUIRED_OUTPUTS),
    "expected_optional_outputs": list(OPTIONAL_OUTPUTS),
    "optional_outputs_present": sorted({f for m in markers.values()
                                        for f in m["written_outputs"]} - set(REQUIRED_OUTPUTS)),
    "run_status_counts": RUN_SUMMARY["status"].value_counts().to_dict(),
    "total_cpdb_seconds": round(float(sum(elapsed_by_run.values())), 3),
    "elapsed_seconds_by_run": elapsed_by_run,
    "run_dirs": {s["name"]: str(s["run_dir"]) for s in SPECS},
    "gate2": {
        "markers_verified": 35,
        "rerun_is_noop": True,
        "pooled_grid_identical": True,
        "precision_note": str(PRECISION_NOTE),
    },
}
SUCCESS_RECORD.write_text(json.dumps(record, indent=2, default=str))
assert SUCCESS_RECORD.is_file() and SUCCESS_RECORD.stat().st_size > 0
print(f"GATE 2 PASSED — {len(SPECS)} runs verified; record at {SUCCESS_RECORD}")